# Dimension 1.2 - Health Indicators: Distribution & Influencing Factors
Panel fixed effects regression with Driscoll-Kraay SEs, Mundlak CRE, SHAP analysis.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve()))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from hdi.config import *
from hdi.data.loaders import load_master_panel
from hdi.models.panel_regression import panel_fixed_effects, mundlak_cre, shap_importance, results_to_table
from hdi.features.transformers import log_transform
from hdi.visualization.themes import apply_theme
apply_theme()

## 1. Panel Fixed Effects: Life Expectancy Determinants
```
LE_it = α_i + γ_t + β₁·ln(GDP_pc_it) + β₂·HealthExp_it + β₃·Education_it + β₄·Urbanization_it + β₅·Sanitation_it + ε_it
```

In [ ]:
panel = load_master_panel()

# Prepare variables
dep_var = 'life_expectancy'
indep_vars_candidates = ['log_gdp_pc_ppp', 'health_exp_pct_gdp', 'literacy_rate', 'urban_pct', 'basic_sanitation_pct']

# Log-transform GDP
if 'gdp_pc_ppp' in panel.columns:
    panel = log_transform(panel, ['gdp_pc_ppp'])

indep_vars = [v for v in indep_vars_candidates if v in panel.columns]

if dep_var in panel.columns and len(indep_vars) >= 2:
    result_fe = panel_fixed_effects(panel, dep_var, indep_vars, cov_type='kernel')
    print(result_fe.summary_text)
else:
    print(f"Need {dep_var} and at least 2 independent variables. Available: {indep_vars}")

## 2. Mundlak CRE Model

In [ ]:
if dep_var in panel.columns and len(indep_vars) >= 2:
    result_cre = mundlak_cre(panel, dep_var, indep_vars)
    print(result_cre.summary_text)

## 3. Comparison Table

In [ ]:
if dep_var in panel.columns and len(indep_vars) >= 2:
    table = results_to_table([result_fe, result_cre])
    display(table)

## 4. SHAP Variable Importance (XGBoost)

In [ ]:
import shap

if dep_var in panel.columns and len(indep_vars) >= 2:
    shap_df, explainer = shap_importance(panel, dep_var, indep_vars)

    # Beeswarm plot
    data = panel[indep_vars].dropna()
    shap.summary_plot(shap_df.values, data, feature_names=indep_vars, show=False)
    plt.title('SHAP Feature Importance for Life Expectancy')
    plt.tight_layout()
    plt.show()

## 5. Distribution by Region/Income (KDE + Violin)

In [ ]:
if 'wb_income' in panel.columns and dep_var in panel.columns:
    fig, ax = plt.subplots(figsize=(10, 6))
    latest = panel[panel['year'] == panel['year'].max()]
    if len(latest) > 0:
        sns.violinplot(data=latest, x='wb_income', y=dep_var, ax=ax,
                      order=['LIC', 'LMC', 'UMC', 'HIC'])
        ax.set_title(f'{dep_var} Distribution by Income Group ({panel["year"].max()})')
        plt.show()